In [75]:
# Standard numpy for matrices, vectors, etc..
import numpy as np

# Visualisation (plotting, etc..)
import matplotlib.pyplot as plt
%matplotlib inline

# New for today! 
# https://pytorch.org/
import torch

# Please see the following page for getting 
# started guide and tutorials:
# https://pytorch.org/tutorials/

In [76]:
# What we are used to in standard programming:

a = np.array([1.0, 2.0, 3.0])

b = np.array([2.0, 2.0, 2.0])

a_plus_b = a + b

a_power_b = a ** b

c = a_plus_b * a_power_b

print('In numpy:')
print('a = ', a)
print('a = ', b)
print('a + b = ', a_plus_b)
print('a ** b = ', a_power_b)
print('c = (a + b) * (a ** b) = ', c)

In numpy:
a =  [1. 2. 3.]
a =  [2. 2. 2.]
a + b =  [3. 4. 5.]
a ** b =  [1. 4. 9.]
c = (a + b) * (a ** b) =  [ 3. 16. 45.]


In [77]:
# Let's do this with PyTorch:
#
# (Using t_ to denote tensor)
#

t_a = torch.tensor(a, requires_grad=True)

t_b = torch.tensor(b, requires_grad=True)

t_a_plus_b = a + b

t_a_power_b = a ** b

t_c = t_a_plus_b * t_a_power_b

print('In PyTorch:')
print('a = ', t_a)
print('a = ', t_b)
print('a + b = ', t_a_plus_b)
print('a ** b = ', t_a_power_b)
print('c = (a + b) * (a ** b) = ', t_c)

In PyTorch:
a =  tensor([1., 2., 3.], dtype=torch.float64, requires_grad=True)
a =  tensor([2., 2., 2.], dtype=torch.float64, requires_grad=True)
a + b =  [3. 4. 5.]
a ** b =  [1. 4. 9.]
c = (a + b) * (a ** b) =  [ 3. 16. 45.]


In [78]:
# Can ignore this code for now, this just creates a 
# visualisation file so we can see what is going on!

from torch.utils.tensorboard import SummaryWriter
import torch.nn as nn

class CustomModel(nn.Module):
    def __init__(self):
        super(CustomModel, self).__init__()

    def forward(self, a, b):
        a_plus_b = a + b
        a_power_b = a ** b
        c = a_plus_b * a_power_b
        return c

model = CustomModel()
summary_writer = SummaryWriter('visualisation_files')

t_a = torch.tensor(a, requires_grad=True)
t_b = torch.tensor(b, requires_grad=True)
example_output = model(t_a, t_b)
summary_writer.add_graph(model, input_to_model=(t_a, t_b))

summary_writer.flush()
summary_writer.close()

# This has created a folder called "visualisation_files" 
# with some data in it that we can view with the command:
#
# tensorboard --logdir=visualisation_files
#

### This has created a computational graph!

<img src="torch_graph_01.png" width="75%">

### Why would we want to do this?

...

In [79]:
# Let's check everything makes sense:

print('In numpy:')
print('a = ', a)
print('a = ', b)
print('a + b = ', a_plus_b)
print('a ** b = ', a_power_b)
print('c = (a + b) * (a ** b) = ', c)


print('In PyTorch:')   
print('a = ', t_a)
print('a = ', t_b)
print('a + b = ', t_a_plus_b)
print('a ** b = ', t_a_power_b)
print('c = (a + b) * (a ** b) = ', t_c)

# Everything should be the same!

In numpy:
a =  [1. 2. 3.]
a =  [2. 2. 2.]
a + b =  [3. 4. 5.]
a ** b =  [1. 4. 9.]
c = (a + b) * (a ** b) =  [ 3. 16. 45.]
In PyTorch:
a =  tensor([1., 2., 3.], dtype=torch.float64, requires_grad=True)
a =  tensor([2., 2., 2.], dtype=torch.float64, requires_grad=True)
a + b =  [3. 4. 5.]
a ** b =  [1. 4. 9.]
c = (a + b) * (a ** b) =  [ 3. 16. 45.]


### So it seems to work!

### But isn't this more effort than the numpy version?

...

### What if we are doing an optimistion?

We want to fit a Gaussian distribution $\mathcal{N}(\mu, \sigma^2)$ to a set of numbers $X = \{ x_0, x_1, \dots, x_{N-1} \}$

If we assume the numbers are i.i.d. (indentically and independently distributed) samples from a Gaussian then the likelihood of $X$ is given by:

\begin{align}
p(X) &= p(x_0) \cdot p(x_1) \cdot \dots \cdot p(x_{N-1}) \\
 &= \mathcal{N}(x_0 \,|\, \mu, \sigma^2) \cdot \mathcal{N}(x_1 \,|\, \mu, \sigma^2) \cdot \dots \cdot \mathcal{N}(x_{N-1} \,|\, \mu, \sigma^2) \\
 &= \prod_{n=0}^{N-1} \mathcal{N}(x_{n} \,|\, \mu, \sigma^2) \\
 &= \prod_{n=0}^{N-1} \frac{1}{\sqrt{2\pi\sigma^2}} 
    \exp{\left( - \frac{(x_{n} - \mu)^2}{2\sigma^2} \right)}
\end{align}

**Top Tip!** When working with exponential family of distributions it often helps to work in the log domain..

\begin{align}
\log  p(X)  &= \sum_{n=0}^{N-1} 
    -\frac{1}{2} \log{\left( 2\pi\sigma^2 \right)}
    -\frac{(x_{n} - \mu)^2}{2\sigma^2}
\end{align}

So, we have the *maximum likelihood* fit to the parameters when we find the values of $\mu$ and $\sigma^2$ that maximise $p(X)$ which (since $\log\,(\cdot)$ is a concave function) occurs at the same time that $\log p(X)$ is maximised.

In our case we can find an analytic solution for

\begin{align}
\mu^{*} &= {\arg\max}_{\mu} \, \log p(X) \\
{\sigma^{*}}^2 &= {\arg\max}_{\sigma^2} \, \log p(X) \\
\end{align}

But let's pretend that the problem was more complicated and we needed to use *optimisation* to solve the problem..

In previous lectures you have seen that you need to be able to calculate gradients of the objective function ($\log p(X)$) wrt the parameters that you are optimising ($\mu$ and $\sigma^2$).

*Let's see how to do this in PyTorch..*

In [80]:
# First let's generate some numbers to fit the data to..

# How many values of x?
N = 20

# Pick the real mean and variance..
mu_true = 2.5
sigma_true = 1.5

x_n = np.random.normal(mu_true, sigma_true, N)

np.set_printoptions(precision=3, linewidth=50)
print('X = \n', np.transpose(x_n))

X = 
 [ 0.081  2.35   4.08   0.562 -0.84  -2.151  0.266
  3.338  2.469  3.476  3.103 -0.107  4.058  1.719
  4.613  5.423  1.379  2.432  1.824  3.865]


We are now going to build our PyTorch graph but we are going to account for the fact that $\mu$ and $\sigma^2$ are no longer constants since we wish to vary their values to find the maximum of $\log p(X)$. As with the optimisation lectures, we need to start with a guess for the values of $\mu$ and $\sigma^2$; in this case, we will start with
\begin{align}
\mu_{\mathrm{initial}} &= 1\\
\sigma^2_{\mathrm{initial}} &= 1
\end{align}

**Top Tip!** Care needs to be taken with $\sigma$ since it can only be a positive value (unlike $\mu$ which can be any real number). In general, PyTorch variables can be positive or negative. In this example we square the value of `t_sigma` before using it to ensure that `t_sigma_2` is a positive value but we shouldn't, therefore, use the value for `t_sigma` directly in calculations..

As a reminder, we want to find:
\begin{align}
\log  p(X)  &= \sum_{n=0}^{N-1} 
    -\frac{1}{2} \log{\left( 2\pi\sigma^2 \right)}
    -\frac{(x_{n} - \mu)^2}{2\sigma^2} 
%    \\
%    &=  -\frac{N}{2} \log{\left( 2\pi\sigma^2 \right)}
%    - \frac{1}{2\sigma^2} \sum_{n=0}^{N-1}\left(x_{n} - \mu\right)^2
\end{align}

In [58]:
# Our initial guesses..
mu_initial_guess = 1.0
sigma_initial_guess = np.sqrt(1.0)

# The data to fit to
t_x_n = torch.tensor(x_n, dtype=torch.float64)

# We need to specify their data type and initial value.. 
t_mu = torch.nn.Parameter(torch.tensor(mu_initial_guess, dtype=torch.float64))
t_sigma = torch.nn.Parameter(torch.tensor(sigma_initial_guess, dtype=torch.float64))

# Note: this step is important - don't use t_sigma directly!! 
t_sigma_2 = t_sigma ** 2.0

# Calculate log p(X) terms..

t_x_minus_mu_2 = (t_x_n - t_mu) ** 2.0
t_denom = 2.0 * t_sigma_2
t_sigma_term = - 0.5 * torch.log(2.0 * np.pi * t_sigma_2)
t_log_P_terms = t_sigma_term - (t_x_minus_mu_2 / t_denom)
t_log_P = torch.sum(t_log_P_terms)

Log probability: -68.61270996301627


In [60]:
# Let's just check that we calculated things correctly:
test_value = t_log_P.item()

# Check with scipy..
from scipy.stats import norm
check_value = np.sum(norm.logpdf(x_n, 
                                 mu_initial_guess, 
                                 sigma_initial_guess))
print('Value from scipy stats package = ', check_value)

assert(np.isclose(test_value, check_value))

print('\nEverything working!')

Value from scipy stats package =  -68.61270996301627

Everything working!


In [61]:
# We can even go crazy and check with different 
# values of the parameters..

mu_new_test_value = 3.3
sigma_new_test_value = 0.5

# We need to specify their data type and initial value.. 
t_mu = torch.nn.Parameter(torch.tensor(mu_new_test_value, dtype=torch.float64))
t_sigma = torch.nn.Parameter(torch.tensor(sigma_new_test_value, dtype=torch.float64))

# Note: this step is important - don't use t_sigma directly!! 
t_sigma_2 = t_sigma ** 2.0

# Calculate log p(X) terms..

t_x_minus_mu_2 = (t_x_n - t_mu) ** 2.0
t_denom = 2.0 * t_sigma_2
t_sigma_term = - 0.5 * torch.log(2.0 * np.pi * t_sigma_2)
t_log_P_terms = t_sigma_term - (t_x_minus_mu_2 / t_denom)
t_log_P = torch.sum(t_log_P_terms)

test_value = t_log_P.item()

# Check with scipy..
from scipy.stats import norm
check_value = np.sum(norm.logpdf(x_n, 
                                 mu_new_test_value, 
                                 sigma_new_test_value))
print('New value from scipy stats package = ', check_value)

assert(np.isclose(test_value, check_value))

print('\nEverything working!')

New value from scipy stats package =  -130.64921429611158

Everything working!


*So finally, we get to the advantage of PyTorch!*

Now, we can calcuate the objective function and we can calculate the value of the objective when changing the input parameters. 

This is great for optimisation (since we are going to need to change the parameters to increase the objective) but what we really need for the optimisation is to calculate the **gradient of the objective wrt to the parameters**. Let's see how to do that in PyTorch..

In [63]:
t_mu = torch.tensor(mu_initial_guess, dtype=torch.float64, requires_grad=True)
t_sigma = torch.tensor(sigma_initial_guess, dtype=torch.float64, requires_grad=True)

t_sigma_2 = t_sigma ** 2.0
t_x_minus_mu_2 = (t_x_n - t_mu) ** 2.0
t_denom = 2.0 * t_sigma_2
t_sigma_term = -0.5 * torch.log(2.0 * np.pi * t_sigma_2)
t_log_P_terms = t_sigma_term - (t_x_minus_mu_2 / t_denom)
t_log_P = torch.sum(t_log_P_terms)

t_log_P.backward()

grad_mu = t_mu.grad
grad_sigma = t_sigma.grad

print('Gradient wrt mu = ', grad_mu.item())
print('Gradient wrt sigma = ', grad_sigma.item())

Gradient wrt mu =  31.130692386138506
Gradient wrt sigma =  80.46787859784563


Shall we check that result. Remember we have:

\begin{align}
\log  p(X)  &= \sum_{n=0}^{N-1} 
    -\frac{1}{2} \log{\left( 2\pi\sigma^2 \right)}
    -\frac{(x_{n} - \mu)^2}{2\sigma^2} \\
    &=  -\frac{N}{2} \log{\left( 2\pi\sigma^2 \right)}
    - \frac{1}{2\sigma^2} \sum_{n=0}^{N-1}\left(x_{n} - \mu\right)^2
\end{align}

So for $\mu$ we have:

\begin{align}
\frac{\partial \log  p(X)}{\partial \mu}   
    &= - 0
    - \frac{1}{2\sigma^2} \frac{\partial}{\partial \mu}  \sum_{n=0}^{N-1} \left(x_{n} - \mu\right)^2 \\
    &= - \frac{1}{2\sigma^2} \sum_{n=0}^{N-1} \frac{\partial}{\partial \mu} \left(x_{n} - \mu\right)^2 \\
    &= - \frac{1}{2\sigma^2} \sum_{n=0}^{N-1} 2 \left(x_{n} - \mu\right) \frac{\partial}{\partial \mu}\left(x_{n} - \mu\right)  \\
    &= \frac{1}{\sigma^2} \sum_{n=0}^{N-1} \left(x_{n} - \mu\right)
\end{align}

where we used the "chain rule" a number of times..

In [64]:
# numpy check of gradient wrt mu

grad_mu_check = np.sum(x_n - mu_initial_guess) / \
                (sigma_initial_guess ** 2)
    
print('Our analytic gradient wrt mu = ', grad_mu_check)

print('PyTorch gradient wrt mu = ', grad_mu.item())

assert(np.isclose(grad_mu.item(), grad_mu_check))

print('\nExcellent! PyTorch calculated the gradient for us :)')

Our analytic gradient wrt mu =  31.1306923861385
PyTorch gradient wrt mu =  31.130692386138506

Excellent! PyTorch calculated the gradient for us :)


## Everyone should now be in awe!

This might seem like something trivial but hopefully you can see that actually quite a lot of maths and then coding went into determining the gradient. 

In fact, you can do the same to check the value for the gradient wrt $\sigma^2$.

When we calculated the result using the chain rule. Since PyTorch built up a graph of the operations, it is able to apply the chain rule results for us automatically.

This:
\begin{align}
\log  p(X)  &= \sum_{n=0}^{N-1} 
    -\frac{1}{2} \log{\left( 2\pi\sigma^2 \right)}
    -\frac{(x_{n} - \mu)^2}{2\sigma^2} 
%    \\
%    &=  -\frac{N}{2} \log{\left( 2\pi\sigma^2 \right)}
%    - \frac{1}{2\sigma^2} \sum_{n=0}^{N-1}\left(x_{n} - \mu\right)^2
\end{align}
has become this:
<img src="graph_02.png" width="50%">

For example, the `pow` operation represents $r = a^b$ for the inputs $a,b$ and result $r$. PyTorch then knows that $\frac{\partial r}{\partial a} = b a^{b-1}$, and by chaining these operations together it can work backwards through the graph (from $\log p(X)$ at the top to $\mu$ at the bottom) to calculate the gradient.

Therefore, the PyTorch graph has multiple uses. A forward pass can calculate the objective for the current set of parameters and a backwards pass can calcuate the gradients of an objective wrt any of the parameters.

## But the fun doesn't end here!

In fact PyTorch has actually done all the work to do the optimisation part, not just calculate the derivatives. So we can now run a full optimisation with our graph and it will use the gradients internally.

In [65]:
# Define parameters mu and sigma with gradients enabled
t_mu = torch.tensor(mu_initial_guess, dtype=torch.float64, requires_grad=True)
t_sigma = torch.tensor(sigma_initial_guess, dtype=torch.float64, requires_grad=True)

# Define the learning rate and optimizer for gradient descent
learning_rate = 0.05
optimizer = torch.optim.SGD([t_mu, t_sigma], lr=learning_rate)

# Objective to minimize is -log P(X), as we want to maximize log P(X)
num_iterations = 50

for iteration in range(num_iterations):
    # Clear gradients from previous step
    optimizer.zero_grad()
    
    # Calculate log P(X)
    t_sigma_2 = t_sigma ** 2.0
    t_x_minus_mu_2 = (t_x_n - t_mu) ** 2.0
    t_denom = 2.0 * t_sigma_2
    t_sigma_term = -0.5 * torch.log(2.0 * np.pi * t_sigma_2)
    t_log_P_terms = t_sigma_term - (t_x_minus_mu_2 / t_denom)
    t_log_P = torch.sum(t_log_P_terms)
    
    # Define the objective as the negative log likelihood
    t_objective = -t_log_P
    
    # Perform backpropagation to calculate gradients
    t_objective.backward()
    
    # Update parameters using the optimizer
    optimizer.step()
    
    # Print log P every 10 iterations
    if (iteration + 1) % 10 == 0:
        print(f'iter {iteration + 1}, log P(X) = {t_log_P.item():.3f}')

# After optimization, print optimized values
print('\nAfter optimization:')
print('Optimized mu =', t_mu.item())
print('Optimized sigma =', t_sigma.item())

# Analytical estimates for comparison
print('\nAnalytic estimates:')
print('Estimated mu =', np.mean(x_n))
print('Estimated std =', np.std(x_n))

print('\nGround truth values:')
print('True mu =', mu_true)
print('True sigma =', sigma_true)

iter 10, log P(X) = -45.271
iter 20, log P(X) = -37.937
iter 30, log P(X) = -37.936
iter 40, log P(X) = -37.936
iter 50, log P(X) = -37.936

After optimization:
Optimized mu = 2.5565346193069254
Optimized sigma = 1.6126357024422246

Analytic estimates:
Estimated mu = 2.5565346193069254
Estimated std = 1.6126357024422246

Ground truth values:
True mu = 2.5
True sigma = 1.5


## Excellent! So we agree with the analytic estimate :)
We can now even make this a function and call it with lots of different data..

In [66]:
def find_parameters_using_pytorch(x_input, mu_initial_guss, sigma_initial_guess, learning_rate=0.05,
                                  num_iterations=50):
    # Convert x_input to a PyTorch tensor
    t_x_n = torch.tensor(x_input, dtype=torch.float64)

    # Initialize parameters with requires_grad=True to track gradients
    t_mu = torch.tensor(mu_initial_guss, dtype=torch.float64, requires_grad=True)
    t_sigma = torch.tensor(sigma_initial_guess, dtype=torch.float64, requires_grad=True)

    # Set up the optimizer with the parameters and specified learning rate
    optimizer = torch.optim.SGD([t_mu, t_sigma], lr=learning_rate)

    # Run the optimization loop for the specified number of iterations
    for iteration in range(num_iterations):
        # Zero the gradients before each iteration
        optimizer.zero_grad()
        
        # Calculate the log probability (log P(X))
        sigma_2 = t_sigma ** 2.0  # Square sigma instead of using it directly
        x_minus_mu_2 = (t_x_n - t_mu) ** 2.0
        denom = 2.0 * sigma_2
        sigma_term = -0.5 * torch.log(2.0 * np.pi * sigma_2)
        log_p_terms = sigma_term - (x_minus_mu_2 / denom)
        t_log_P = torch.sum(log_p_terms)  # Sum to get scalar log probability

        # Define the objective as the negative log likelihood to maximize log P(X)
        t_objective = -t_log_P

        # Perform backpropagation to compute gradients
        t_objective.backward()

        # Update parameters using the optimizer
        optimizer.step()

        # Print the value of log P every 10 iterations
        if (iteration + 1) % 10 == 0:
            print(f'iter {iteration + 1}, log P(X) = {t_log_P.item():.3f}')

    # Return the optimized values of mu and sigma
    return t_mu.item(), t_sigma.item()

In [83]:
# Let's try with a larger N

N_bigger = 1000

x_bigger = np.random.normal(mu_true, sigma_true, N_bigger)

new_mu, new_sigma = find_parameters_using_pytorch(x_bigger, mu_initial_guess, sigma_initial_guess, 
                                                  learning_rate=0.001, num_iterations=50)

print('PyTorch estimates:')
print('PyTorch mu = ', new_mu)
print('PyTorch sigma = ', new_sigma)

print('\nAnalytic estimates:')
print('Estimated mu = ', np.mean(x_bigger))
print('Estimated std = ', np.std(x_bigger))

print('\nGround truth values:')
print('True mu = ', mu_true)
print('True sigma = ', sigma_true)

iter 10, log P(X) = -2139.400
iter 20, log P(X) = -1852.691
iter 30, log P(X) = -1852.691
iter 40, log P(X) = -1852.691
iter 50, log P(X) = -1852.691
PyTorch estimates:
PyTorch mu =  2.513788803091006
PyTorch sigma =  1.5430368402036756

Analytic estimates:
Estimated mu =  2.5137888030910056
Estimated std =  1.5430368402036754

Ground truth values:
True mu =  2.5
True sigma =  1.5


## All sorts of more advanced topics

- Visualise parts of computation (e.g. Tensorboard)
- Reusable components (e.g. modules for neural networks / classifiers / etc..)
- Run computations on the GPU instead of the CPU (often faster)
- Easy to scale; can distribute computations over an entire cluster!